# Explainable AML monitoring: run the pipeline in order

This notebook walks through the code one section at a time. The saved **IBM data profile** output is a snapshot from the full downloaded CSV. The feature and model teaching sections use a **small artificial demo**. The final section reads a separate **10% sampled IBM pilot** with real labels and complete prior transaction history; it is not a full-dataset experiment.

Follow the [README setup and data-download steps](../README.md) before a fresh **Run All**. Select this repository's `.venv/bin/python` kernel. Without local generated results, the IBM profile and sampled-study cells display reproduction commands instead of tables. For the source-code reading order, see [`docs/START_HERE.md`](../docs/START_HERE.md).

## What this notebook covers

The full project plan has **14 sections**. This notebook executes the core pipeline on an artificial demo, includes saved full IBM profile output, and ends with saved summary output from a separate 10% sampled IBM modeling pilot. A fresh run reads the locally generated profile and study JSON files; they are not in Git.

| Project section | Status in this notebook |
| --- | --- |
| 1. Dataset and exploratory analysis | Saved full IBM profile and sampled study summary |
| 2. Databricks and dbt | Planned |
| 3. Behavioral features | Demo execution; currency-consistent histories in code; sampled study with complete prior history |
| 4. Rule-based monitoring | Demo rule and sampled-study currency-aware rule |
| 5. Logistic regression, XGBoost, EBM | Demo execution and saved sampled-study results |
| 6. Deep learning | Planned |
| 7. Graph-based detection | Planned |
| 8. Model metrics | Demo execution; sampled average precision and alert metrics |
| 9. SHAP and EBM explanations | Sampled-study artifacts; local Tree SHAP |
| 10. Validation and backtesting | Demo split; sampled calendar holdout and late stress period |
| 11. Monitoring and MLOps | Saved daily replay counts; formal drift thresholds planned |
| 12. Transaction replay and streaming | Saved-score replay; live scoring planned |
| 13. Investigation dashboard | Separate local Streamlit interface |
| 14. Investigation narratives and governance | Initial sampled-study report and diagnostics |

See [`docs/PROJECT_SCOPE.md`](../docs/PROJECT_SCOPE.md) for the deliverables and status.

## 0. Setup

Find the project root, check that VS Code selected this repository's `.venv` kernel and required packages, then import the same functions used by the CLI. The first code cell stops with instructions if another environment is selected.

In [1]:
from pathlib import Path
import importlib.util
import json
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the AML project folder")

PROJECT_ENV = PROJECT_ROOT / ".venv"
if Path(sys.prefix).resolve() != PROJECT_ENV.resolve():
    raise RuntimeError(
        f"Wrong notebook kernel: {sys.executable}\n"
        "In VS Code, click the kernel name at the top right, choose "
        "Select Another Kernel -> Python Environments, then select "
        f"{PROJECT_ENV / 'bin' / 'python'}. Restart the kernel and Run All."
    )

required = ("numpy", "pandas", "sklearn", "xgboost", "interpret", "IPython")
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        f"Missing packages in the project environment: {missing}. "
        "From the project folder, run: .venv/bin/python -m pip install -e '.[dev,notebook]'"
    )

import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from aml_monitoring.data import make_demo_transactions
from aml_monitoring.evaluation import chronological_split, score_predictions
from aml_monitoring.features import FEATURES, build_features
from aml_monitoring.models import fit_model, rule_scores

SEED = 42
DEMO_ROWS = 1200
ALERT_CAPACITY = 25
print("Kernel: project .venv")
print(f"Project: {PROJECT_ROOT.name}")
print(f"Demo: {DEMO_ROWS:,} artificial rows; alert capacity: {ALERT_CAPACITY}")

Kernel: project .venv
Project: Explainable-AML-Transaction-Monitoring
Demo: 1,200 artificial rows; alert capacity: 25


## 1. Inspect the real IBM data profile

The profile was produced by `.venv/bin/aml --csv data/raw/HI-Small_Trans.csv --profile-only --output results/ibm`. This cell reads the **locally generated** JSON instead of loading 5 million rows again. The JSON is ignored by Git; a fresh clone must download and profile the CSV first. The saved output below is a snapshot from the original workspace.

The late-period label shift is a reason to settle the test design before reporting real model scores.

In [2]:
profile_path = PROJECT_ROOT / "results" / "ibm" / "data_profile.json"
if profile_path.exists():
    ibm_profile = json.loads(profile_path.read_text())
    display(pd.DataFrame([{
        "transactions": ibm_profile["rows"],
        "labeled_laundering": ibm_profile["positives"],
        "label_rate_pct": round(100 * ibm_profile["positive_rate"], 4),
        "first_timestamp": ibm_profile["first_timestamp"],
        "last_timestamp": ibm_profile["last_timestamp"],
    }]))
    split_profile = pd.DataFrame(ibm_profile["splits"]).T
    display(split_profile[["rows", "positives", "positive_rate", "first_timestamp", "last_timestamp"]])
    daily_profile = pd.DataFrame(ibm_profile["daily_activity"]).T
    daily_profile["label_rate_pct"] = 100 * daily_profile["positives"] / daily_profile["rows"]
    print("Last eight days of the IBM file:")
    display(daily_profile.tail(8))
else:
    ibm_profile = None
    print("IBM profile not found. Run the profile command in the text above, then rerun this cell.")

,transactions,labeled_laundering,label_rate_pct,first_timestamp,last_timestamp
0,5078345,5177,0.1019,2022-09-01 00:00:00,2022-09-18 16:18:00


,rows,positives,positive_rate,first_timestamp,last_timestamp
train,3047216,2299,0.000754,2022-09-01 00:00:00,2022-09-06 13:36:00
validation,1015565,1081,0.001064,2022-09-06 13:37:00,2022-09-08 16:12:00
test,1015564,1797,0.001769,2022-09-08 16:13:00,2022-09-18 16:18:00


Last eight days of the IBM file:


,rows,positives,label_rate_pct
2022-09-11,396,232,58.585859
2022-09-12,281,170,60.498221
2022-09-13,184,106,57.608696
2022-09-14,121,70,57.851240
2022-09-15,46,28,60.869565
2022-09-16,46,26,56.521739
2022-09-17,23,15,65.217391
2022-09-18,11,8,72.727273


## 2. Load example transactions (`data.py`)

`make_demo_transactions()` gives us a small, reproducible dataset in the IBM column format. Its labels are deliberately easy to learn so this section checks the code path, not research performance.

In [3]:
transactions = make_demo_transactions(n=DEMO_ROWS, seed=SEED)
print(f"Rows: {len(transactions):,}; labeled positives: {transactions['is_laundering'].sum():,}")
display(transactions[[
    "timestamp", "from_bank", "from_account", "to_bank", "to_account",
    "amount_paid", "payment_currency", "payment_format", "is_laundering",
]].head(5))

Rows: 1,200; labeled positives: 87


,timestamp,from_bank,from_account,to_bank,to_account,amount_paid,payment_currency,payment_format,is_laundering
0,2024-01-01 00:00:00,002,S0004,001,R0048,605.380337,USD,ACH,0
1,2024-01-01 00:30:00,002,S0034,003,R0026,425.069330,USD,ACH,0
2,2024-01-01 01:00:00,002,S0029,001,R0054,2507.515129,USD,ACH,0
3,2024-01-01 01:30:00,002,S0019,003,R0041,280.541117,USD,ACH,0
4,2024-01-01 02:00:00,002,S0019,003,R0035,10817.162278,USD,Wire,1


## 3. Split by time (`evaluation.py`)

Training uses the earliest rows, validation the middle rows, and test the latest rows. Rows with an identical timestamp stay together. The test labels are shown here for learning purposes, but the models fit only on training labels.

In [4]:
split = chronological_split(transactions["timestamp"])
split_summary = pd.DataFrame([
    {
        "period": period,
        "rows": len(indexes),
        "positives": int(transactions.iloc[indexes]["is_laundering"].sum()),
        "first": transactions.iloc[indexes]["timestamp"].min(),
        "last": transactions.iloc[indexes]["timestamp"].max(),
    }
    for period, indexes in split.items()
])
display(split_summary)

,period,rows,positives,first,last
0,train,720,44,2024-01-01,2024-01-15 23:30:00
1,validation,240,21,2024-01-16,2024-01-20 23:30:00
2,test,240,22,2024-01-21,2024-01-25 23:30:00


## 4. Build point-in-time features (`features.py`)

Each row gets current-transaction fields plus the sender's earlier 24-hour and 7-day activity. Transactions at the same time cannot see one another. We build the chronological history across periods because earlier transactions are known when later ones are scored; labels are never used in feature construction.

In [5]:
features = build_features(transactions)
print(f"Feature matrix: {features.shape[0]:,} rows × {features.shape[1]} columns")
display(features[[
    "log_amount_paid", "prior_txn_count_24h", "prior_amount_24h",
    "prior_txn_count_7d", "prior_avg_amount_7d", "new_counterparty_7d",
    "payment_format",
]].head(5))
prior_rows = features.index[features["prior_txn_count_24h"] > 0]
if len(prior_rows):
    row = int(prior_rows[0])
    print(f"Example row with prior activity: {row}")
    display(features.loc[[row], FEATURES])

Feature matrix: 1,200 rows × 13 columns


,log_amount_paid,prior_txn_count_24h,prior_amount_24h,prior_txn_count_7d,prior_avg_amount_7d,new_counterparty_7d,payment_format
0,6.407507,0,0.000000,0,0.000000,1,ACH
1,6.054602,0,0.000000,0,0.000000,1,ACH
2,7.827446,0,0.000000,0,0.000000,1,ACH
3,5.640279,0,0.000000,0,0.000000,1,ACH
4,9.288982,1,280.541117,1,280.541117,1,Wire


Example row with prior activity: 4


,log_amount_paid,hour,day_of_week,same_bank,prior_txn_count_24h,prior_amount_24h,prior_txn_count_7d,prior_avg_amount_7d,amount_vs_prior_avg_7d,new_counterparty_7d,payment_format,payment_currency,receiving_currency
4,9.288982,2,0,0,1,280.541117,1,280.541117,38.558206,1,Wire,USD,USD


## 5. Fit the four baselines (`models.py`)

The fixed rules require no fitting. Logistic regression, XGBoost, and EBM fit on **training rows only**. Categorical values are encoded inside the logistic and XGBoost pipelines; EBM reads the named columns directly.

In [6]:
labels = transactions["is_laundering"].to_numpy()
trained = {}
for name in ("logistic", "xgboost", "ebm"):
    trained[name] = fit_model(name, features.iloc[split["train"]][FEATURES],
                              labels[split["train"]], seed=SEED)
    print(f"Fitted {name} on {len(split['train']):,} training rows")
print("Rules: fixed thresholds; no training")

Fitted logistic on 720 training rows
Fitted xgboost on 720 training rows


Fitted ebm on 720 training rows
Rules: fixed thresholds; no training


## 6. Score and evaluate (`evaluation.py`)

We compare validation and test ranking quality and how many positive labels appear within the highest-scored `K` alerts. The `pr_auc` JSON field is calculated with scikit-learn's **average precision** function, a summary of the precision–recall curve. `recall_at_k` measures detections at a fixed review capacity. Demo figures below have no research meaning.

In [7]:
metrics = {}
prediction_parts = []
for name in ("rules", "logistic", "xgboost", "ebm"):
    metrics[name] = {}
    for period in ("validation", "test"):
        indexes = split[period]
        x = features.iloc[indexes][FEATURES]
        scores = rule_scores(x) if name == "rules" else trained[name].predict_proba(x)[:, 1]
        metrics[name][period] = score_predictions(labels[indexes], scores, ALERT_CAPACITY)
        prediction_parts.append(pd.DataFrame({
            "source_row": indexes,
            "timestamp": transactions.iloc[indexes]["timestamp"].to_numpy(),
            "period": period,
            "model": name,
            "label": labels[indexes],
            "score": scores,
        }))
predictions = pd.concat(prediction_parts, ignore_index=True)
comparison = pd.DataFrame([
    {"model": model, "period": period, **values}
    for model, periods in metrics.items()
    for period, values in periods.items()
])
display(comparison[[
    "model", "period", "pr_auc", "roc_auc", "precision_at_k",
    "recall_at_k", "positives_found", "alerts",
]].round(4))

,model,period,pr_auc,roc_auc,precision_at_k,recall_at_k,positives_found,alerts
0,rules,validation,0.8248,0.9631,0.72,0.8571,18,25
1,rules,test,0.8408,0.9656,0.68,0.7727,17,25
2,logistic,validation,1.0000,1.0000,0.84,1.0000,21,25
3,logistic,test,1.0000,1.0000,0.88,1.0000,22,25
4,xgboost,validation,1.0000,1.0000,0.84,1.0000,21,25
5,xgboost,test,1.0000,1.0000,0.88,1.0000,22,25
6,ebm,validation,1.0000,1.0000,0.84,1.0000,21,25
7,ebm,test,1.0000,1.0000,0.88,1.0000,22,25


## 7. Look at the highest-ranked demo transactions

This is a first look at individual scores. It is **not yet a SHAP or EBM explanation**; those are later research steps. A high score means the model ranked a transaction for review, not that it proved laundering.

In [8]:
top = predictions.query("model == 'xgboost' and period == 'test'").nlargest(5, "score")
examples = transactions.loc[top["source_row"], [
    "timestamp", "from_account", "to_account", "amount_paid",
    "payment_currency", "payment_format", "is_laundering",
]].copy()
examples["xgboost_score"] = top["score"].to_numpy()
display(examples)

,timestamp,from_account,to_account,amount_paid,payment_currency,payment_format,is_laundering,xgboost_score
1027,2024-01-22 09:30:00,S0012,R0051,18628.863323,USD,Wire,1,0.998334
1028,2024-01-22 10:00:00,S0025,R0015,12344.826719,USD,Wire,1,0.998334
1058,2024-01-23 01:00:00,S0019,R0000,29615.678908,USD,Wire,1,0.998334
1075,2024-01-23 09:30:00,S0007,R0023,27588.505206,USD,Wire,1,0.998334
1121,2024-01-24 08:30:00,S0003,R0001,18362.742561,USD,Wire,1,0.998334


## 8. Save this notebook's results

The notebook already contains the visible outputs above. This cell also writes machine-readable demo metrics and scores under ignored `results/notebook_demo/` so you can inspect them beside the CLI outputs.

In [9]:
output_dir = PROJECT_ROOT / "results" / "notebook_demo"
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
predictions.to_csv(output_dir / "predictions.csv", index=False)
print("Saved: results/notebook_demo/metrics.json")
print("Saved: results/notebook_demo/predictions.csv")

Saved: results/notebook_demo/metrics.json
Saved: results/notebook_demo/predictions.csv


## From demo to the sampled IBM pilot

The full IBM profile shows a sharp shift after September 10: only 1,108 later transactions, 655 of them positive. The pilot uses September 1–10 for its primary comparison, keeps the late period separate, and samples 10% of primary rows for model fitting and evaluation. DuckDB computes behavioral history from **all** earlier original transactions before those modeling rows are selected. See the [study report](../docs/STUDY_RESULTS.md) before interpreting any score.

## 9. Inspect the sampled real-data study

This cell reads the separate real-data study's local metrics and diagnostics. Its saved output is a snapshot from the original workspace. To regenerate it, install `.[full-history]`, run `.venv/bin/python -m aml_monitoring.study --csv data/raw/HI-Small_Trans.csv --fraction 0.10 --full-history --output results/study_full_history_10pct`, then `.venv/bin/python -m aml_monitoring.diagnostics --study-dir results/study_full_history_10pct --repeats 300`. The study reads the entire CSV and uses disk-backed windows.

In [10]:
study_dir = PROJECT_ROOT / 'results' / 'study_full_history_10pct'
if (study_dir / 'diagnostics.json').exists():
    study_manifest = json.loads((study_dir / 'study_manifest.json').read_text())
    study_metrics = json.loads((study_dir / 'metrics.json').read_text())
    study_diagnostics = json.loads((study_dir / 'diagnostics.json').read_text())
    chosen = study_manifest['selected_by_validation_average_precision']
    print(f"Sample: {study_manifest['sample_fraction_primary']:.0%} of the primary period; "
          f"{study_manifest['periods']['test']['rows']:,} primary test rows and "
          f"{study_manifest['periods']['test']['positives']} positive labels.")
    print(f'Validation-selected model: {chosen}')
    print(f"Validation AP: {study_metrics[chosen]['validation']['pr_auc']:.4f}")
    print(f"Primary test AP: {study_metrics[chosen]['test']['pr_auc']:.4f}")
    print(f"Test labels found in top 100: {study_diagnostics['true_positives_at_capacity']}")
    low, high = study_diagnostics['stratified_bootstrap_95pct_interval']
    print(f'Descriptive bootstrap AP interval: {low:.4f}–{high:.4f}')
    print('Full details: docs/STUDY_RESULTS.md')
else:
    print('Sampled study outputs not found. Run the two study commands in the section above.')

Sample: 10% of the primary period; 86,066 primary test rows and 103 positive labels.
Validation-selected model: xgboost_behavioral
Validation AP: 0.3362
Primary test AP: 0.2245
Test labels found in top 100: 23
Descriptive bootstrap AP interval: 0.1468–0.3128
Full details: docs/STUDY_RESULTS.md


## 10. Review the saved-score alert replay

The replay selects a cutoff from validation **scores only**, then processes saved test and late-stress scores in timestamp order. It illustrates alert volume and drift without pretending to score a live stream. Generate it with `.venv/bin/python -m aml_monitoring.replay --study-dir results/study_full_history_10pct --daily-capacity 100`.

In [11]:
replay_path = PROJECT_ROOT / 'results' / 'study_full_history_10pct' / 'replay_summary.json'
if replay_path.exists():
    replay = json.loads(replay_path.read_text())
    print(f"Validation-only cutoff: {replay['validation_score_cutoff']:.6f}")
    for period, values in replay['replay_periods'].items():
        print(f"{period}: {values['rows']:,} rows, {values['alerts']:,} alerts, "
              f"{values['labels_in_alerts']:,} labels in alerts")
else:
    print('Replay output not found. Run the replay command above, then rerun this cell.')

Validation-only cutoff: 0.949697
late_stress: 1,108 rows, 388 alerts, 361 labels in alerts
test: 86,066 rows, 221 alerts, 24 labels in alerts


## Remaining full-data work

The sampled pilot is a class-project milestone with complete prior history for the selected rows. Before claiming full-dataset model performance, fit and evaluate on all eligible rows, validate the late-period shift, repeat the time-window analysis, calibrate scores, and review case-level explanations. The [scope map](../docs/PROJECT_SCOPE.md) tracks the larger engineering and dashboard sections.